# EEG_15 — Li et al.: Trial-Level Hypergraph + Label Propagation

**Approccio diametralmente opposto a EEG_13**:

| Aspetto | EEG_13 (DHSLP nostro) | EEG_15 (Li et al. spirit) |
|---------|----------------------|--------------------------|
| Vertici del grafo | Elettrodi (61) | **Trial** (~35K) |
| Iperedge | Apprese da backprop (learnable E) | Costruite da similarità in feature space |
| Feature nodo | Segnale grezzo finestrato | Gamma-band features pre-estratte |
| Inferenza | Inductive (nuovi soggetti) | **Transductive** (tutti i trial nel grafo) |
| Supervision | Supervised (label per ogni trial) | **Semi-supervised** (label propagation) |

**Pipeline EEG_15**:
```
Per ogni trial: x (61, 384)
  → filtraggio gamma (30-80 Hz)
  → split K=12 finestre temporali
  → mean power per canale per finestra  → (61×12 = 732,)
  → PCA → 128D
                     ↓
Tutti i trial come nodi: grafo N_trials × N_trials
  → KMeans clustering in feature space → H (N_trials × N_edges)
  → Laplaciano dell'ipergrafo Θ
                     ↓
Label Propagation (Zhu 2003 / Zhou 2004):
  F* = (I - α Θ)^{-1} Y_0
  → predict argmax F*[test_trials]
```

**Riferimento**: Li et al. 2025 — EEG-based speech imagery decoding via dynamic hypergraph learning.

In [ ]:
import json, logging, re, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from scipy import signal as scipy_signal
from scipy.sparse import csr_matrix, diags as sp_diags
from scipy.sparse.linalg import spsolve
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg15')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents)
                     if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg15'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS   = 61
N_SAMPLES    = 384
SFREQ        = 256
N_CLASSES    = 4
CLUSTER_SCHEME = 'concr4'

# Feature extraction
GAMMA_LOW   = 30    # Hz
GAMMA_HIGH  = 80    # Hz
K_WIN_FEAT  = 12    # finestre temporali per feature extraction (vs K=8 di EEG_13)
# → 12 × 61 = 732 feature grezze per trial → PCA → PCA_COMPONENTS

PCA_COMPONENTS = 128  # come Li et al.

# Ipergrafo
N_HG_EDGES_LIST = [32, 64, 128]  # ablation
LP_ALPHA         = 0.9            # coefficiente label propagation
LP_MAX_ITER      = 200

# Split soggetti (subject-independent — identico a EEG_09–13)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Sorgente dati: stesso PT di EEG_13 (x grezzo + label)
DATA_METRIC = 'abs_pcc'

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

# Raccolta path
HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess[int(m.group(1))][int(m.group(2))].append(p)

log.info(f'Soggetti trovati: {len(subj_sess)}')
log.info(f'SUBJ_TRAIN={len(SUBJ_TRAIN)} SUBJ_VAL={len(SUBJ_VAL)} SUBJ_TEST={len(SUBJ_TEST)}')

## §2 — Feature Extraction per Trial (Gamma-Band)

Per ogni trial: bandpass gamma (30–80 Hz) → split K=12 finestre → mean power per canale → (732,).  
Cache su disco: `figures/eeg15_trial_features.npz` (~35K × 732).

In [ ]:
FEAT_CACHE = FIG_DIR / 'eeg15_trial_features.npz'

# ── Butterworth bandpass filter ───────────────────────────────────────────────
_b_gamma, _a_gamma = scipy_signal.butter(
    4, [GAMMA_LOW, GAMMA_HIGH], btype='bandpass', fs=SFREQ)

def extract_gamma_features(x_np):
    """
    x_np: (61, 384) float32
    Output: (732,) = 61 canali × 12 finestre temporali (mean gamma power)
    """
    # Bandpass gamma
    x_filt = scipy_signal.filtfilt(_b_gamma, _a_gamma, x_np, axis=1)  # (61, 384)
    # Instantaneous power (envelope²)
    power = x_filt ** 2   # (61, T)
    # Split in K_WIN_FEAT finestre uguali
    win_len = N_SAMPLES // K_WIN_FEAT   # 32 campioni
    feats = []
    for k in range(K_WIN_FEAT):
        seg = power[:, k*win_len:(k+1)*win_len]   # (61, win_len)
        feats.append(seg.mean(axis=1))             # (61,)
    return np.concatenate(feats)   # (732,)


if FEAT_CACHE.exists():
    log.info(f'Carico feature da cache: {FEAT_CACHE}')
    cache = np.load(FEAT_CACHE, allow_pickle=True)
    ALL_FEATURES = cache['features']   # (N, 732)
    ALL_LABELS   = cache['labels']     # (N,) cluster id
    ALL_SUBJ     = cache['subjects']   # (N,) subject id
    ALL_SPLIT    = cache['splits']     # (N,) 0=train, 1=val, 2=test
    log.info(f'Feature matrix: {ALL_FEATURES.shape}')
else:
    log.info('Estrazione feature gamma... (prima esecuzione ~5-10 min)')
    feat_list, lbl_list, subj_list, split_list = [], [], [], []

    split_map = {**{s: 0 for s in SUBJ_TRAIN},
                 **{s: 1 for s in SUBJ_VAL},
                 **{s: 2 for s in SUBJ_TEST}}

    for sid in tqdm(sorted(subj_sess.keys()), desc='Soggetti'):
        sp = split_map.get(sid, -1)
        if sp < 0:
            continue
        for sess_id, paths in subj_sess[sid].items():
            for p in paths:
                try:
                    d = torch.load(p, weights_only=False)
                    x_np = d['x'].float().numpy()  # (61, 384)
                    y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
                    c = label2cluster.get(y_word)
                    if c is None:
                        continue
                    feat = extract_gamma_features(x_np)
                    feat_list.append(feat)
                    lbl_list.append(c)
                    subj_list.append(sid)
                    split_list.append(sp)
                except Exception as e:
                    log.warning(f'Errore {p}: {e}')
                    continue

    ALL_FEATURES = np.stack(feat_list, axis=0).astype(np.float32)  # (N, 732)
    ALL_LABELS   = np.array(lbl_list,  dtype=np.int32)
    ALL_SUBJ     = np.array(subj_list, dtype=np.int32)
    ALL_SPLIT    = np.array(split_list, dtype=np.int32)

    np.savez(FEAT_CACHE, features=ALL_FEATURES, labels=ALL_LABELS,
             subjects=ALL_SUBJ, splits=ALL_SPLIT)
    log.info(f'Salvato: {FEAT_CACHE}  shape={ALL_FEATURES.shape}')

N_TRIALS = len(ALL_FEATURES)
TRAIN_MASK = ALL_SPLIT == 0
VAL_MASK   = ALL_SPLIT == 1
TEST_MASK  = ALL_SPLIT == 2

log.info(f'Totale trial: {N_TRIALS} — train={TRAIN_MASK.sum()} val={VAL_MASK.sum()} test={TEST_MASK.sum()}')
print(f'Label distribution train: {np.bincount(ALL_LABELS[TRAIN_MASK])}')
print(f'Label distribution test:  {np.bincount(ALL_LABELS[TEST_MASK])}')

## §3 — Normalizzazione + PCA → 128D

StandardScaler fit su TRAIN, transform su tutti. PCA 128 componenti.

In [ ]:
# Normalizzazione (fit su train only)
scaler = StandardScaler()
FEAT_SCALED = scaler.fit_transform(ALL_FEATURES)   # (N, 732)
log.info('StandardScaler fit su TRAIN — transform su tutti i trial')

# PCA → 128D
n_comp = min(PCA_COMPONENTS, FEAT_SCALED.shape[1], FEAT_SCALED.shape[0]-1)
pca = PCA(n_components=n_comp, random_state=42)
pca.fit(FEAT_SCALED[TRAIN_MASK])  # fit su solo TRAIN
FEAT_PCA = pca.transform(FEAT_SCALED).astype(np.float32)   # (N, 128)

var_explained = pca.explained_variance_ratio_.sum()
log.info(f'PCA {n_comp} componenti → {var_explained*100:.1f}% varianza spiegata (fit su train)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
axes[0].plot(np.cumsum(pca.explained_variance_ratio_) * 100, linewidth=2)
axes[0].axhline(90, color='red', linestyle='--', label='90%')
axes[0].axhline(var_explained*100, color='green', linestyle='--',
                label=f'{var_explained*100:.1f}% ({n_comp} comp)')
axes[0].set_xlabel('Componenti PCA'); axes[0].set_ylabel('Varianza Cumulativa (%)')
axes[0].set_title('PCA — Varianza Spiegata'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Distribuzione primi 2 PCA per split
for mask, label, color in [(TRAIN_MASK, 'Train', '#2C7BB6'),
                             (VAL_MASK,   'Val',   '#FD8D3C'),
                             (TEST_MASK,  'Test',  '#D7191C')]:
    idx = np.where(mask)[0]
    sample = idx[np.random.choice(len(idx), min(500, len(idx)), replace=False)]
    axes[1].scatter(FEAT_PCA[sample, 0], FEAT_PCA[sample, 1],
                   c=color, label=label, alpha=0.4, s=10)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA — Distribuzione trial per split'); axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg15_pca.png', dpi=150); plt.show()
log.info(f'Salvato: {FIG_DIR / "eeg15_pca.png"}')

## §4 — Costruzione Ipergrafo (Trial come Vertici)

Ogni trial = nodo. Iperedge = cluster di trial simili nello spazio PCA.

**Metodo**: KMeans con N_EDGES cluster → H[i,j] = 1 se trial_i ∈ cluster_j.  
H ∈ {0,1}^{N_trials × N_edges} — incidence matrix sparsa.

**Laplaciano dell'ipergrafo** (Zhou et al. 2006):
```
Θ = D_v^{-1/2} H W D_e^{-1} H^T D_v^{-1/2}
```
dove W = I (pesi uniformi), D_v = diag(H @ 1), D_e = diag(H^T @ 1).

In [ ]:
def build_hypergraph(feat_pca, n_edges, seed=42):
    """
    Costruisce H (N_trials, n_edges) via KMeans sul feature space PCA.
    Ritorna: H sparse (N, n_edges), Theta = prop matrix (N, N) sparse.
    """
    log.info(f'KMeans: n_edges={n_edges} ...')
    t0 = time.time()
    km = KMeans(n_clusters=n_edges, n_init=5, max_iter=300, random_state=seed)
    assignments = km.fit_predict(feat_pca)   # (N,) ∈ [0, n_edges)

    # H binaria: riga=trial, col=cluster
    N = len(feat_pca)
    rows = np.arange(N)
    H = csr_matrix((np.ones(N, dtype=np.float32), (rows, assignments)),
                   shape=(N, n_edges))   # sparse (N, n_edges)

    # D_e: degree iperedge (quanti trial per cluster)
    d_e = np.asarray(H.sum(axis=0)).flatten()  # (n_edges,)
    D_e_inv = sp_diags(1.0 / np.maximum(d_e, 1e-8))   # (n_edges, n_edges)

    # D_v: degree nodo (quante iperedge per trial) = 1 per costruzione
    d_v = np.asarray(H.sum(axis=1)).flatten()   # (N,)
    D_v_invsqrt = sp_diags(1.0 / np.maximum(np.sqrt(d_v), 1e-8))

    # Θ = D_v^{-1/2} H D_e^{-1} H^T D_v^{-1/2}
    Theta = D_v_invsqrt @ H @ D_e_inv @ H.T @ D_v_invsqrt

    log.info(f'  Ipergrafo: {N} trial × {n_edges} edge — {time.time()-t0:.1f}s')
    return H, Theta


# Prova con N_HG_EDGES_LIST[1] (default = 64) per sanity check
H_default, Theta_default = build_hypergraph(FEAT_PCA, N_HG_EDGES_LIST[1])

# Visualizza distribuzione dimensioni cluster
km_tmp = KMeans(n_clusters=N_HG_EDGES_LIST[1], n_init=5, random_state=42)
asgn = km_tmp.fit_predict(FEAT_PCA)
cluster_sizes = np.bincount(asgn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(cluster_sizes)), sorted(cluster_sizes, reverse=True))
axes[0].set_xlabel('Iperedge (rank per dimensione)'); axes[0].set_ylabel('# Trial')
axes[0].set_title(f'Distribuzione dimensioni cluster (N_edges={N_HG_EDGES_LIST[1]})')
axes[0].grid(axis='y', alpha=0.3)

# Composizione split per cluster
for k_edges in N_HG_EDGES_LIST:
    km_viz = KMeans(n_clusters=k_edges, n_init=3, random_state=42)
    a = km_viz.fit_predict(FEAT_PCA)
    train_frac = [((a==c) & TRAIN_MASK).sum() / max((a==c).sum(), 1) for c in range(k_edges)]
    axes[1].plot(sorted(train_frac), label=f'N_edges={k_edges}')
axes[1].set_xlabel('Cluster rank'); axes[1].set_ylabel('Frazione trial TRAIN')
axes[1].set_title('Purezza cluster (% trial da TRAIN)')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(TRAIN_MASK.mean(), linestyle='--', color='k', label='TRAIN baseline')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg15_hypergraph_stats.png', dpi=150); plt.show()
log.info(f'Salvato: {FIG_DIR / "eeg15_hypergraph_stats.png"}')

## §5 — Label Propagation (Semi-Supervised)

**Formula** (Zhu & Ghahramani 2003 / Zhou et al. 2004 per ipergrafi):
```
F_{t+1} = α Θ F_t + (1 - α) Y_0
F* = (I - α Θ)^{-1} Y_0   (closed-form)
```

- `Y_0[i]` = one-hot label se trial_i ∈ TRAIN ∪ VAL, 0 se test  
- `α = 0.9` — quanto peso alla propagazione vs inizializzazione  
- `F*[i]` = label scores propagati → `argmax F*[i]` = predizione

**Ablation**: N_HG_EDGES ∈ {32, 64, 128}

In [ ]:
def label_propagation(Theta, labels, train_mask, alpha=LP_ALPHA, max_iter=LP_MAX_ITER):
    """
    Label propagation sull'ipergrafo.
    Theta: (N, N) sparse — matrice di propagazione
    labels: (N,) int — cluster id (ignorati per nodi non-train)
    train_mask: (N,) bool — nodi con label note
    Ritorna: F_star (N, N_CLASSES) float — score propagati
    """
    N = len(labels)
    # Y_0: one-hot per i nodi labeled
    Y_0 = np.zeros((N, N_CLASSES), dtype=np.float32)
    for i in np.where(train_mask)[0]:
        Y_0[i, labels[i]] = 1.0

    # Iterative label propagation
    F = Y_0.copy()
    for it in range(max_iter):
        F_new = alpha * (Theta @ F) + (1 - alpha) * Y_0
        delta = np.abs(F_new - F).max()
        F = F_new
        if delta < 1e-6:
            log.info(f'  LP converged @ iter {it+1} (delta={delta:.2e})')
            break
    return F   # (N, N_CLASSES)


# Ablation N_HG_EDGES
RESULTS_LP = {}

for n_edges in N_HG_EDGES_LIST:
    log.info(f'\n=== Label Propagation — N_edges={n_edges} ===')
    t0 = time.time()

    H, Theta = build_hypergraph(FEAT_PCA, n_edges)

    # LP su TRAIN mask (TRAIN+VAL come labeled)
    labeled_mask = TRAIN_MASK | VAL_MASK
    F_star = label_propagation(Theta, ALL_LABELS, labeled_mask)

    # Valutazione su TEST
    test_idx   = np.where(TEST_MASK)[0]
    test_preds = F_star[test_idx].argmax(axis=1)
    test_gt    = ALL_LABELS[test_idx]
    val_idx    = np.where(VAL_MASK)[0]
    val_preds  = F_star[val_idx].argmax(axis=1)
    val_gt     = ALL_LABELS[val_idx]

    test_bacc = balanced_accuracy_score(test_gt, test_preds)
    val_bacc  = balanced_accuracy_score(val_gt,  val_preds)

    # Anche su TRAIN (solo per debug — non è train acc, è just propagation residual)
    train_idx   = np.where(TRAIN_MASK)[0]
    train_preds = F_star[train_idx].argmax(axis=1)
    train_bacc  = balanced_accuracy_score(ALL_LABELS[train_idx], train_preds)

    elapsed = time.time() - t0
    RESULTS_LP[n_edges] = {
        'test_bacc': test_bacc, 'val_bacc': val_bacc, 'train_bacc': train_bacc,
        'test_preds': test_preds, 'test_gt': test_gt, 'time': elapsed
    }
    log.info(f'  N_edges={n_edges}: val={val_bacc:.4f}  test={test_bacc:.4f}  ({elapsed:.1f}s)')

print('\n── Label Propagation Results ──')
chance = 1 / N_CLASSES
for ne, res in RESULTS_LP.items():
    above = '✓' if res['test_bacc'] > chance else '✗'
    print(f'  N_edges={ne:3d}: val={res["val_bacc"]:.4f}  test={res["test_bacc"]:.4f}  {above} (chance={chance:.2%})')

## §6 — Baseline: k-NN nel Feature Space PCA

Confronto con semplice k-NN (non usa ipergrafo) per separare il contributo dell'ipergrafo da quello delle feature.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

RESULTS_KNN = {}
for k in [5, 15, 30]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1)
    # Fit su TRAIN+VAL
    labeled_mask = TRAIN_MASK | VAL_MASK
    knn.fit(FEAT_PCA[labeled_mask], ALL_LABELS[labeled_mask])

    val_preds  = knn.predict(FEAT_PCA[VAL_MASK])
    test_preds = knn.predict(FEAT_PCA[TEST_MASK])

    val_bacc  = balanced_accuracy_score(ALL_LABELS[VAL_MASK],  val_preds)
    test_bacc = balanced_accuracy_score(ALL_LABELS[TEST_MASK], test_preds)
    RESULTS_KNN[k] = {'val_bacc': val_bacc, 'test_bacc': test_bacc}
    log.info(f'  k-NN k={k}: val={val_bacc:.4f}  test={test_bacc:.4f}')

print('\n── k-NN Baseline ──')
for k, res in RESULTS_KNN.items():
    print(f'  k={k:2d}: val={res["val_bacc"]:.4f}  test={res["test_bacc"]:.4f}')

## §7 — Visualizzazioni + Confronto con EEG_13

1. Confusion matrix del miglior modello LP
2. Tabella comparativa EEG_15 vs altri notebook

In [ ]:
import matplotlib.gridspec as gridspec

chance = 1 / N_CLASSES

# Miglior configurazione LP
best_ne   = max(RESULTS_LP, key=lambda k: RESULTS_LP[k]['val_bacc'])
best_res  = RESULTS_LP[best_ne]
best_knn  = max(RESULTS_KNN, key=lambda k: RESULTS_KNN[k]['val_bacc'])

print(f'Best LP: N_edges={best_ne} → val={best_res["val_bacc"]:.4f}  test={best_res["test_bacc"]:.4f}')
print(f'Best kNN: k={best_knn} → val={RESULTS_KNN[best_knn]["val_bacc"]:.4f}  test={RESULTS_KNN[best_knn]["test_bacc"]:.4f}')

fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# ── Panel 1: ablation LP ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
edges = list(RESULTS_LP.keys())
val_b  = [RESULTS_LP[e]['val_bacc']  for e in edges]
test_b = [RESULTS_LP[e]['test_bacc'] for e in edges]
x_ = np.arange(len(edges))
ax1.bar(x_-0.2, val_b,  0.35, label='Val',  color='#FD8D3C', alpha=0.9)
ax1.bar(x_+0.2, test_b, 0.35, label='Test', color='#2C7BB6', alpha=0.9)
ax1.axhline(chance, color='k', linestyle='--', linewidth=1.5, label=f'Chance ({chance:.0%})')
ax1.set_xticks(x_)
ax1.set_xticklabels([f'N_e={e}' for e in edges])
ax1.set_ylabel('Balanced Accuracy')
ax1.set_title('EEG_15 — Label Propagation\nAblation N_edges')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# ── Panel 2: Confusion Matrix miglior LP ─────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
cm = confusion_matrix(best_res['test_gt'], best_res['test_preds'], normalize='true')
im = ax2.imshow(cm, cmap='Blues', vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax2.text(j, i, f'{cm[i,j]:.2f}', ha='center', va='center', fontsize=10,
                color='white' if cm[i,j] > 0.5 else 'black')
ax2.set_xticks(range(N_CLASSES))
ax2.set_yticks(range(N_CLASSES))
LABEL_NAMES = ['CONCR', 'AZIONE', 'STATO', 'ASTRATTO']
ax2.set_xticklabels(LABEL_NAMES, rotation=30)
ax2.set_yticklabels(LABEL_NAMES)
ax2.set_title(f'Confusion Matrix (LP, N_edges={best_ne})\ntest bAcc={best_res["test_bacc"]:.4f}')
fig.colorbar(im, ax=ax2, fraction=0.046)

# ── Panel 3: Confronto con altri notebook ────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
# Risultati noti da altri notebook (Subject-Independent, concr4)
comparison = {
    'HGNN\n(EEG_09)':     0.257,
    'T-HGNN\n(EEG_10)':   0.248,
    'W-HGNN\n(EEG_11)':   0.255,
    'DHSLP\n(EEG_13)':    None,   # da inserire se disponibile
    'kNN\nbaseline':       RESULTS_KNN[best_knn]['test_bacc'],
    f'LP N_e={best_ne}\n(EEG_15)': best_res['test_bacc'],
}
# Rimuovi None
comparison = {k: v for k, v in comparison.items() if v is not None}

keys  = list(comparison.keys())
vals  = list(comparison.values())
cols  = ['#2ca02c' if v > chance + 0.005 else '#d62728' for v in vals]
# Evidenzia EEG_15
cols[-1] = '#1f77b4'   # blu per il nuovo metodo
ax3.bar(keys, vals, color=cols, alpha=0.85)
ax3.axhline(chance, color='k', linestyle='--', linewidth=1.5, label=f'Chance ({chance:.0%})')
ax3.set_ylabel('Balanced Accuracy (test)')
ax3.set_title('Confronto Metodi\n(Subject-Independent, concr4)')
ax3.legend(); ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(0, max(vals) + 0.03)

plt.tight_layout()
out = FIG_DIR / 'eeg15_results.png'
plt.savefig(out, dpi=150); plt.show()
log.info(f'Salvato: {out}')

## §8 — W&B Logging

In [ ]:
wandb.login()

for n_edges, res in RESULTS_LP.items():
    run = wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name=f'eeg15_LP_E{n_edges}_{CLUSTER_SCHEME}',
        config=dict(
            notebook='EEG_15', model='LabelPropagation',
            approach='trial_level_hypergraph',
            n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
            gamma_low=GAMMA_LOW, gamma_high=GAMMA_HIGH,
            k_win_feat=K_WIN_FEAT, pca_components=PCA_COMPONENTS,
            n_hg_edges=n_edges, lp_alpha=LP_ALPHA,
            n_train_trials=int(TRAIN_MASK.sum()),
            n_val_trials=int(VAL_MASK.sum()),
            n_test_trials=int(TEST_MASK.sum()),
        ),
        reinit='finish_previous',
        settings=wandb.Settings(start_method='thread')
    )
    run.log({
        'val/bacc':  res['val_bacc'],
        'test/bacc': res['test_bacc'],
        'train/bacc': res['train_bacc'],
        'elapsed_s': res['time'],
    })
    run.summary['val_bacc']  = res['val_bacc']
    run.summary['test_bacc'] = res['test_bacc']
    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
        preds=res['test_preds'].tolist(),
        y_true=res['test_gt'].tolist(),
        class_names=['CONCR', 'AZIONE', 'STATO', 'ASTRATTO'])})
    run.finish()

# Log anche kNN baseline
for k, res in RESULTS_KNN.items():
    run = wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name=f'eeg15_kNN_k{k}_{CLUSTER_SCHEME}',
        config=dict(notebook='EEG_15', model='kNN', k=k,
                    gamma_low=GAMMA_LOW, gamma_high=GAMMA_HIGH,
                    k_win_feat=K_WIN_FEAT, pca_components=PCA_COMPONENTS,
                    n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME),
        reinit='finish_previous',
        settings=wandb.Settings(start_method='thread')
    )
    run.summary['val_bacc']  = res['val_bacc']
    run.summary['test_bacc'] = res['test_bacc']
    run.finish()

log.info('W&B logging completato')

# ── Tabella riassuntiva finale ────────────────────────────────────────────────
print('\n' + '='*55)
print('EEG_15 — RIEPILOGO FINALE')
print('='*55)
print(f'Feature: gamma {GAMMA_LOW}-{GAMMA_HIGH}Hz × {K_WIN_FEAT} win × {N_CHANNELS}ch → PCA-{PCA_COMPONENTS}')
print(f'Approccio: trial-level hypergraph + label propagation')
print(f'n_trials totali: {N_TRIALS}  (train={TRAIN_MASK.sum()} val={VAL_MASK.sum()} test={TEST_MASK.sum()})')
print()
print('Label Propagation:')
for ne, res in sorted(RESULTS_LP.items()):
    mark = '← best' if ne == best_ne else ''
    print(f'  N_edges={ne:3d}: val={res["val_bacc"]:.4f}  test={res["test_bacc"]:.4f}  {mark}')
print()
print('k-NN baseline:')
for k, res in RESULTS_KNN.items():
    mark = '← best' if k == best_knn else ''
    print(f'  k={k:2d}: val={res["val_bacc"]:.4f}  test={res["test_bacc"]:.4f}  {mark}')
print()
print(f'Chance level: {chance:.4f} ({chance:.0%})')